In [3]:
# 한국어 BERT vs GPT 출력 비교 실습 (KLUE BERT / KoGPT2)
from transformers import BertTokenizer, BertModel, GPT2TokenizerFast, GPT2LMHeadModel
import torch
import torch.nn.functional as F

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 입력 문장
text = "나는 오늘 아침에 사과를 먹었다."
print("\n✅ 입력 문장:", text)
print("=" * 60)

# BERT: KLUE BERT 문맥 임베딩 출력
print("\n🔵 [KLUE BERT] 문맥 임베딩 출력:")
bert_tokenizer = BertTokenizer.from_pretrained("klue/bert-base")
bert_model = BertModel.from_pretrained("klue/bert-base").to(device)
bert_model.eval()

inputs_bert = bert_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs_bert = bert_model(**inputs_bert)

tokens_bert = bert_tokenizer.convert_ids_to_tokens(inputs_bert['input_ids'][0])
for i, token in enumerate(tokens_bert):
    vec = outputs_bert.last_hidden_state[0][i]
    print(f"{token:10} → 임베딩 평균값: {vec.mean().item():.4f}")

print("\n" + "-" * 60)

# GPT: KoGPT2 다음 단어 예측
print("\n🟠 [KoGPT2] 다음 단어 예측 (Top-3):")

gpt_tokenizer = GPT2TokenizerFast.from_pretrained("skt/kogpt2-base-v2")
gpt_model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2").to(device)
gpt_model.eval()

inputs_gpt = gpt_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs_gpt = gpt_model(**inputs_gpt)

tokens_gpt = gpt_tokenizer.convert_ids_to_tokens(inputs_gpt['input_ids'][0])
logits = outputs_gpt.logits

def strip_sp(t):
    t2 = t.replace('▁', '')
    return t2 if t2.strip() else "[공백]"

for i in range(len(tokens_gpt) - 1):
    token = tokens_gpt[i]
    next_logits = logits[0, i]                 # 위치 i가 예측한 '다음 토큰'의 점수들
    probs = F.softmax(next_logits, dim=-1)     # 확률로 변환
    topk = torch.topk(probs, k=3)              # 상위 3개
    top_tokens = gpt_tokenizer.convert_ids_to_tokens(topk.indices.tolist())
    top_probs = topk.values.tolist()
    print(f"{strip_sp(token):10} → 다음 예측: "
          f"{strip_sp(top_tokens[0])}({top_probs[0]:.2f}), "
          f"{strip_sp(top_tokens[1])}({top_probs[1]:.2f}), "
          f"{strip_sp(top_tokens[2])}({top_probs[2]:.2f})")

print("\n" + "-" * 60)



✅ 입력 문장: 나는 오늘 아침에 사과를 먹었다.

🔵 [KLUE BERT] 문맥 임베딩 출력:
[CLS]      → 임베딩 평균값: 0.0067
나          → 임베딩 평균값: 0.0128
##는        → 임베딩 평균값: 0.0107
오늘         → 임베딩 평균값: 0.0126
아침         → 임베딩 평균값: 0.0120
##에        → 임베딩 평균값: 0.0116
사과         → 임베딩 평균값: 0.0108
##를        → 임베딩 평균값: 0.0128
먹          → 임베딩 평균값: 0.0116
##었        → 임베딩 평균값: 0.0110
##다        → 임베딩 평균값: 0.0116
.          → 임베딩 평균값: 0.0121
[SEP]      → 임베딩 평균값: 0.0100

------------------------------------------------------------

🟠 [KoGPT2] 다음 단어 예측 (Top-3):
나는         → 다음 예측: "(0.01), 이(0.01), 지난(0.01)
오늘         → 다음 예측: 도(0.18), 은(0.07), 이(0.06)
아침에        → 다음 예측: 일어나(0.02), 그(0.02), [공백](0.01)
사          → 다음 예측: 과를(0.13), 십(0.03), 온(0.03)
과를         → 다음 예측: 했(0.08), 드(0.08), 하고(0.07)
먹          → 다음 예측: 었(0.23), 었는데(0.09), 으러(0.08)

------------------------------------------------------------
